# Phase 7 : Projections de Prévisions 2030 & Simulation ROI Hôtelier Marrakech

Ce notebook contient la phase finale de projection et d'interprétation économique :
1. **Projections d'Arrivées 2026-2030** à long terme (56 mois).
2. **Estimation des Recettes Touristiques** avec prise en compte de l'inflation cumulée et du boost Coupe du Monde (+20% en 2030).
3. **Simulation financière d'un hôtel** 5 étoiles (200 chambres, $40M USD de coût d'investissement) à Marrakech, comparant le scénario de base et le scénario avec l'effet d'attractivité de la Coupe du Monde (calcul de NPV & IRR).

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('..'))
from src.config import DATA_DIR, TARGET_COL
from main import forecast_recursive_ml, forecast_recursive_dl

from src.models.random_forest import RandomForestModel
from src.models.lstm import LstmModel
from src.models.sarima import SarimaModel

## 1. Chargement et Entraînement Global

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, 'merged_tourism_data_final.csv'))
df['Date'] = pd.to_datetime(df['Date'])

df_ml = df.dropna(subset=[TARGET_COL]).copy()

X_train = df_ml.drop(columns=['Date', TARGET_COL, 'Total_Receipts_MDH'], errors='ignore')
y_train = df_ml[TARGET_COL]

# Remplissage des NaNs
median_val = X_train.median()
X_train = X_train.fillna(median_val)
valid_features = list(X_train.columns)

# Entraînement sur tout l'historique pour la projection
best_ml = RandomForestModel().fit(X_train, y_train)
best_dl = LstmModel(epochs=10).fit(X_train, y_train)
best_stat = SarimaModel().fit(y_train)

print("Modèles ajustés avec succès sur l'ensemble de l'historique !")

## 2. Projections à l'Horizon 2030 (Arrivées & Recettes)

Nous projetons les 56 prochains mois (mai 2026 à décembre 2030).

In [ ]:
steps = 56
future_dates = pd.date_range(start='2026-05-01', periods=steps, freq='MS')

arrivals_ml = forecast_recursive_ml(best_ml, df_ml, future_dates, valid_features)
arrivals_dl = forecast_recursive_dl(best_dl, df_ml, future_dates, valid_features)
arrivals_sarima = best_stat.predict(steps=steps)

print("Projections d'arrivées générées !")

## 3. Calcul des Recettes Touristiques avec Inflation et Boost CDM 2030

Nous appliquons l'inflation de 1.5% par an et le boost Coupe du Monde 2030 de +20% sur les dépenses moyennes.

In [ ]:
mean_ratio = (df_ml['Total_Receipts_MDH'] / df_ml['Arrivals']).mean()
inflation_rate = 0.015
world_cup_boost = 0.20

def project_receipts(arrivals):
    receipts = []
    for arr, date in zip(arrivals, future_dates):
        years_since_2026 = date.year - 2026
        ratio = mean_ratio * ((1 + inflation_rate) ** years_since_2026)
        if date.year == 2030:
            ratio = ratio * (1 + world_cup_boost)
        receipts.append(arr * ratio)
    return np.array(receipts)

receipts_ml = project_receipts(arrivals_ml)
print(f"Recettes attendues cumulées en 2030 (ML) : {receipts_ml[-12:].sum():.2f} MDH")

## 4. Simulation du ROI d'un Hôtel de Référence à Marrakech

Nous mesurons l'attractivité du secteur hôtelier à travers l'NPV et l'IRR de notre investissement de 40M USD.

In [ ]:
import main
logger = main.setup_logging(DATA_DIR)
df_roi = main.run_hotel_roi_simulation(logger)
df_roi.head(10)